Görkem Kadir Solun

# Assignment 2 - RAG (and a bit of Reasoning)

Parts which require your interaction are marked with `TODO:`

In [1]:
# make sure we have the relevant dependencies
!pip3 install datasets sentence_transformers tqdm numpy

Imagine your task is to build a question-answering (QA) system for a company. You are given a language model and have to create this product out of it.
The requirements of the system need to adapt very quickly to the new data without training.
For this, we will use Retrieval Augmented Generation (RAG).
The company insists you use their in-house LM model trained on multiple tasks, a _flan-t5-small_.
You can test its QA functionality by asking the question _"When ETH was founded?"_:

In [2]:
# There seems to be an issue with pipelines so we have this custom function.
from transformers import T5ForConditionalGeneration, T5Tokenizer

model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-small")
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-small")
def vanilla_qa_pipe(text: str) -> str:
  inputs=tokenizer(text, return_tensors="pt")
  outputs = model.generate(**inputs, max_new_tokens=20)
  return tokenizer.decode(outputs[0], skip_special_tokens=True)

QUESTION = "When was ETH founded?"
vanilla_qa_pipe(f"QUESTION:{QUESTION} ANSWER:")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


...

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

'1897'

In [3]:
vanilla_qa_pipe(f"""
        CONTEXT: ETH Zurich (German: Eidgenoessische Technische Hochschule Zurich; English:
        Federal Institute of Technology Zurich) is a public research university in Zurich,
        Switzerland. Founded in 1854 with the stated mission to educate engineers and scientists,
        the university focuses primarily on science, technology, engineering, and mathematics. It
        consistently ranks among the top universities in the world and its 16 departments span a
        variety of disciplines and subjects.
        {QUESTION}
        ANSWER:",
    """,
)

'1854'

The first output is 1897, which is incorrect.

In the second example, the model was given context from which it could answer the question. So it gave the right answer. However, if you think of it, it was quite lucky that the context contained the answer it was looking for.

If you are not convinced, look at what happens when we change the question:

In [4]:
QUESTION = "When was EPFL founded?"
vanilla_qa_pipe(f"""
        CONTEXT: ETH Zurich (German: Eidgenoessische Technische Hochschule Zurich; English:
        Federal Institute of Technology Zurich) is a public research university in Zurich,
        Switzerland. Founded in 1854 with the stated mission to educate engineers and scientists,
        the university focuses primarily on science, technology, engineering, and mathematics. It
        consistently ranks among the top universities in the world and its 16 departments span a
        variety of disciplines and subjects.
        {QUESTION}
        ANSWER:",
    """,
)

'1854'

It would be nice if both federal institutes were founded in the same year, but they were not. EPFL was actually established in 1969. Now the obvious question would be, "Why not give the first paragraph of **EPFL's** wikipedia page as context?

Well, yes, that would actually work. But for the company's QA system, you cannot sit there looking at every query, searching for the relevant info and feed in.

You do know that customers will only ever ask about things the company works on, and all relevant info is in the company database.

Q1. So why don't we just feed the model the entire database? (0.5 point)

A1. The model has a fixed context window (flan-t5-small accepts at most 512 input tokens), so a real company knowledge base will not fit at all. Even on a large-context model, transformer self-attention is quadratic in the input length, so packing thousands of irrelevant documents into every prompt would make inference prohibitively slow and expensive. On top of that, irrelevant content may hurt answer quality. The model gets distracted, attention mass is spread thinly, and useful evidence gets lost. Restricting the context to the most relevant snippets is what makes RAG both feasible and accurate.


Okay, so we need something better. We earlier talked about how *you* cannot search for the context. But that does not mean *searching* was a bad idea. You could write code to automatically search the database and find the context. But what would you search for? Well, the only thing we have is the question, so might as well go with that.

In [5]:
# You only need to run this if you want to use nltk
import nltk
nltk.download('punkt_tab')
from nltk import word_tokenize
# You might need this
word_tokenize("Tokenize this. See what happens.")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


['Tokenize', 'this', '.', 'See', 'what', 'happens', '.']

In [6]:

class basic_search_engine:
    def __init__(self, dataset) -> None:
        self.dataset=list(dataset)
    def get_score(self, question_counts: dict, target: str) -> int:
        # count how many words are shared between the question and the target
        # if a word appears k times in the question and n times in the target
        # it should contribute k*n to the score.
        # Q2.1 TODO (1 points)
        target_tokens = word_tokenize(target.lower())
        target_counts: dict = {}
        for w in target_tokens:
            target_counts[w] = target_counts.get(w, 0) + 1
        score = 0
        for word, k in question_counts.items():
            if word in target_counts:
                score += k * target_counts[word]
        return score

    def search(self, question: str, count: int) -> list[str]:
        # return the top <count> contexts in terms of score
        # Q2.2 TODO (1 points)
        question_tokens = word_tokenize(question.lower())
        question_counts: dict = {}
        for w in question_tokens:
            question_counts[w] = question_counts.get(w, 0) + 1
        scores = [self.get_score(question_counts, t) for t in self.dataset]
        ranked = sorted(range(len(self.dataset)), key=lambda i: scores[i], reverse=True)
        return [self.dataset[i] for i in ranked[:count]]

# Testing
data=["All cats can fly", "Some cats can fly planes", "Dogs fly higher than cats", "Dogs can not fly planes"]
question="Can dogs fly?"
search_engine=basic_search_engine(data)
search_engine.search(question, 2)

['Dogs can not fly planes', 'All cats can fly']

If it worked with flying dogs, it should work with the real data. Test it out.

In [7]:
import tqdm
import numpy as np
from datasets import load_dataset
dataset = load_dataset("rajpurkar/squad")
context_set=set(dataset["validation"]["context"])
print("Total number of contexts is", len(context_set))
search_engine=basic_search_engine(context_set)

def metric_exact_match(ans_pred: str, ans_true: str) -> float:
    return (ans_pred == ans_true)*1.0

def metric_f1(ans_pred: str, ans_true: str) -> float:
    import collections
    ans_pred = ans_pred.lower().split()
    ans_true = ans_true.lower().split()
    common = collections.Counter(ans_pred) & collections.Counter(ans_true)
    num_same = sum(common.values())

    if num_same == 0:
        return num_same

    prec = num_same/len(ans_pred)
    recl = num_same/len(ans_true)
    return 2*prec*recl/(prec+recl)

exact_match_scores={}
f1_scores={}
exact_match_scores["search_1"]=[]
f1_scores["search_1"]=[]
for line in tqdm.tqdm(dataset["validation"].select(range(100))):
  question=line["question"]
  contexts=search_engine.search(question, 1)
  model_output=vanilla_qa_pipe(f"CONTEXT: {contexts[0]}\nQUESTION:{question}\nANSWER:")
  exact_match_scores["search_1"].append(max([metric_exact_match(model_output, a) for a in line["answers"]["text"]]))
  f1_scores["search_1"].append(max([metric_f1(model_output, a) for a in line["answers"]["text"]]))
print()
print("Average Exact Match Score:", np.average(exact_match_scores["search_1"]))
print("Average F1 Score:", np.average(f1_scores["search_1"]))

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Total number of contexts is 2067


100%|██████████| 100/100 [03:33<00:00,  2.13s/it]


Average Exact Match Score: 0.03
Average F1 Score: 0.07133333333333333


Well, Looks like that was neither fast, nor useful. The fact is, searching like this is not quite optimal.

Q3. So what are some issues with the search? Explain with insights from the ETH example (1 point)

A3.

- It is purely lexical, not semantic. The score only counts surface word overlap, so synonyms and paraphrases miss completely. For "When was EPFL founded?" any context that talks about something being established, opened or created will score zero on that informative word, while an unrelated paragraph that happens to share the verb "founded" (e.g., "Founded in 1854 ... ETH Zurich") will rise to the top. That is how the ETH paragraph ends up being retrieved as the best context for an EPFL question.
- Common words dominate the score. Stopwords like "when", "was", "the", "is" appear in almost every context and contribute heavily to the score even though they carry no information about the question. There is no down weighting of frequent, uninformative tokens (which is the gap that TF-IDF fills).
- There is length bias. A Wikipedia paragraph naturally contains more tokens, so its expected overlap with any question is higher. Useful short contexts are penalised.
- No notion of named entities or topicality. The word ETH or EPFL in the query should match the topic of the context, but it is treated identically to a generic verb like founded. A context about EPFL might score lower than a context about ETH that happens to repeat stopwords from the question.
- Also, it is inefficient. The score is recomputed against every document with a Python loop, which scales as O(|DB| * |context|) per query and is a non-starter on a real database.


Searching a database like this is a non-trivial task and a whole research field of Information Retrieval is devoted to it.

For now, we will pick two off-the-shelf methods, TF-IDF Vectorization aand Embedding with Bert.

You might want to look up what these do. Then answer all the following in 1-3 sentences (0.5 point each)

Q4.1. What is TF-IDF?

A4.1. TF-IDF (Term Frequency – Inverse Document Frequency) is a classic word weighting scheme that scores each (word, document) pair as `tf(w, d) * idf(w)`, where `tf` measures how often the word appears in the document and `idf = log(N / df(w))` down weights words that appear in many documents. The result is that words which are frequent in this document but rare across the whole corpus get the highest weight, so a TF-IDF vector emphasises a documents distinctive terms while suppressing stopwords.

Q4.2. What is the benefit of vectorisation?

A4.2. Once every document and every query is represented as a fixed size numeric vector, similarity becomes a single matrix vector dot product, which is muuch faster than comparing strings token by token and is trivially batchable on a GPU. It also lets us reuse the whole geometric / linear algebra toolbox like cosine similarity, nearest-neighbour indexes, clustering, dimensionality reduction. None of which are available on raw text.

Q4.3. What is a sentence embedding?

A4.3. A sentence embedding is a single dense vector that summarises the meaning of a whole sentence, typically produced by a transformer and trained so that semantically similar sentences land close together in the embedding space, regardless of whether they share any surface words. This lets a search engine retrieve "EPFL was established in 1969" for the query "When was EPFL founded?" even though the two sentences have almost no token overlap.

Now complete the code below. We have already filled in the initialization


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=768, norm=None)

class tfidf_search_engine:
    def __init__(self, dataset) -> None:
      self.vectorizer = TfidfVectorizer(max_features=768, norm=None)
      self.dataset=list(dataset)
      self.kb_vectorized = self.vectorizer.fit_transform([x for x in self.dataset])
    def search(self, question: str) -> list[str]:
        # return the top contexts in terms of tf-idf score
        # we use cosine similarity, so score(a,b)=vector(a).vector(b)
        # you can use the "transform" function of self.vectorizer
        # q5. TODO (1 point)
        q_vec = self.vectorizer.transform([question])           # sparse: [1, vocab]
        scores = (self.kb_vectorized @ q_vec.T).toarray().ravel()  # [n_dataset]
        best_idx = int(np.argmax(scores))
        return self.dataset[best_idx]


# Testing
data=["All cats can fly", "Some cats can fly planes", "Dogs fly higher than cats", "Dogs can not fly planes"]
question="Can dogs fly?"
search_engine=tfidf_search_engine(data)
search_engine.search(question)

'Dogs can not fly planes'

In [9]:
exact_match_scores["search_tfidf"]=[]
f1_scores["search_tfidf"]=[]
tfidf_engine=tfidf_search_engine(context_set)
for line in tqdm.tqdm(dataset["validation"].select(range(100))):
  question=line["question"]
  contexts=tfidf_engine.search(question)
  model_output=vanilla_qa_pipe(f"CONTEXT: {contexts}\nQUESTION:{question}\nANSWER:")
  exact_match_scores["search_tfidf"].append(max([metric_exact_match(model_output, a) for a in line["answers"]["text"]]))
  f1_scores["search_tfidf"].append(max([metric_f1(model_output, a) for a in line["answers"]["text"]]))
print()
print("Average Exact Match Score:", np.average(exact_match_scores["search_tfidf"]))
print("Average F1 Score:", np.average(f1_scores["search_tfidf"]))

100%|██████████| 100/100 [00:35<00:00,  2.80it/s]


Average Exact Match Score: 0.05
Average F1 Score: 0.12533333333333332


In [10]:
from sentence_transformers import SentenceTransformer

class sbert_search_engine:
    def __init__(self, dataset) -> None:
      self.vectorizer = SentenceTransformer("bert-base-nli-mean-tokens").cuda()
      self.dataset=list(dataset)
      self.kb_vectorized = self.vectorizer.encode(self.dataset)
    def search(self, question: str) -> list[str]:
        # return the top contexts in terms of bert-similarity
        # we use cosine similarity, so score(a,b)=vector(a).vector(b)
        # you can use the "encode" function of self.vectorizer
        # q6. TODO (1 point)
        q_vec = self.vectorizer.encode(question)           # [dim]
        scores = self.kb_vectorized @ q_vec                # [n_dataset]
        best_idx = int(np.argmax(scores))
        return self.dataset[best_idx]


# Testing
data=["All cats can fly", "Some cats can fly planes", "Dogs fly higher than cats", "Dogs can not fly planes"]
question="Can dogs fly?"
search_engine=sbert_search_engine(data)
search_engine.search(question)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

'Dogs fly higher than cats'

In [11]:
exact_match_scores["search_sbert"]=[]
sbert_engine=sbert_search_engine(context_set)
f1_scores["search_sbert"]=[]
for line in tqdm.tqdm(dataset["validation"].select(range(100))):
  question=line["question"]
  contexts=sbert_engine.search(question)
  model_output=vanilla_qa_pipe(f"CONTEXT: {contexts}\nQUESTION:{question}\nANSWER:")
  exact_match_scores["search_sbert"].append(max([metric_exact_match(model_output, a) for a in line["answers"]["text"]]))
  f1_scores["search_sbert"].append(max([metric_f1(model_output, a) for a in line["answers"]["text"]]))
print()
print("Average Exact Match Score:", np.average(exact_match_scores["search_sbert"]))
print("Average F1 Score:", np.average(f1_scores["search_sbert"]))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/bert-base-nli-mean-tokens
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 100/100 [00:24<00:00,  4.04it/s]


Average Exact Match Score: 0.17
Average F1 Score: 0.18266666666666664


You would note that the scores are still not that high. Perhaps because the similarity metrics are not perfect.

Q7. Calculate how often the top scoring context is in fact the correct context for a) TF-IDF and b) sBert Score (you would have to write some code for this. Only work with the first 100 validation questions) (1.5 point)

A7. Running the cell below on the first 100 SQuAD validation questions gives:

- TF-IDF top-1 accuracy: 0.00 (0 / 100)
- SBERT top-1 accuracy: 0.12 (12 / 100)

Both numbers are low, but they are consistent with what we saw in the QA experiments (cell 17 EM = 0.05 with TF-IDF vs. cell 19 EM = 0.17 with SBERT) and they explain why the QA scores are so weak. The retriever almost never lands on the gold context.

Two effects combine to produce these numbers:

1. The vectorizer is configured with `norm=None`. Without L2 normalization the score is just a raw dot product, which is dominated by document length and by the absolute frequency of the matched terms. As a result the TF-IDF retriever returns the same handful of long contexts for almost every question, and on a 2067-context knowledge base that handful never coincides with the question context. Hence the 0/100. Switching to `norm="l2"` or applying cosine normalization manually would lift this score.
2. The SBERT checkpoint is `bert-base-nli-mean-tokens`, an older NLI-trained encoder that was not specifically optimized for asymmetric question and passage retrieval. It still beats TF-IDF on this task confirming the qualitative point that dense semantic embeddings capture paraphrases TF-IDF cannot. But a modern retrieval-tuned model such as `all-MiniLM-L6-v2` would push this number much higher.

SBERT > TF-IDF (0.12 > 0.00), and the gap is the headroom that downstream QA accuracy is missing.



In [12]:
# write your code for Q7 here
# The 'correct' context for the question dataset["validation"]["question"][i] is
# dataset["validation"]["context"][i]
matches_tf_idf=0
matches_sbert=0
for i in tqdm.tqdm(range(100)):
  if dataset["validation"]["context"][i]==tfidf_engine.search(dataset["validation"]["question"][i]):
    matches_tf_idf+=1
  if dataset["validation"]["context"][i]==sbert_engine.search(dataset["validation"]["question"][i]):
    matches_sbert+=1
print("TF-IDF:", matches_tf_idf/100)
print("SBERT:", matches_sbert/100)


100%|██████████| 100/100 [00:02<00:00, 49.54it/s]

TF-IDF: 0.0
SBERT: 0.12


Q8. What can we do to account for imperfect similarity metrics? (Hint: if you have been following the code, there was already a hint somewhere) (0.5 point)

A8. The hint is the `count` argument of `basic_search_engine.search`. Instead of returning only the highest scoring context, retrieve the top-k candidates and feed all of them to the QA model in one prompt. As long as the correct context is anywhere in the top-k (which, per Q7, happens more often than it is exactly rank 1), the model can attend to it and produce the right answer. I believe, this is the standard retrieve‑then‑read recipe used by real RAG systems, and it can be combined with reranking (run an expensive cross-encoder on the k candidates) to push the right context to the very top.


You may have noticed that sometimes model can give the right answer despite having the wrong context. But the reverse can also be true.

Q9. For each of the following examples, explain why the model is getting the answer wrong (0.5 point each)

In [13]:
context="The Berlin Wall (German: Berliner Mauer) was a guarded concrete barrier that encircled West Berlin from 1961 to 1989, separating it from East Berlin and the German Democratic Republic (GDR; East Germany). Construction of the Berlin Wall was commenced by the government of the GDR on 13 August 1961. It included guard towers placed along large concrete walls,[4] accompanied by a wide area (later known as the \"death strip\") that contained anti-vehicle trenches, beds of nails and other defenses. The primary intention for the Wall's construction was to prevent East German citizens from fleeing to the West"
question="For many years did the Berlin wall stand?"
vanilla_qa_pipe(f"CONTEXT: {context}\nQUESTION:{question}\nANSWER:")

'a guarded concrete barrier'



A9.1: The model returned "a guarded concrete barrier", which is the very first noun phrase the context uses to predicate something about the Berlin Wall ("The Berlin Wall ... was a guarded concrete barrier that encircled West Berlin from 1961 to 1989"). The reason this happens is that the question is grammatically malformed. It should read "For how many years did the Berlin Wall stand?". Without the wh-word "how" the prompt is no longer recognisable as a duration question. flan-t5-small parses "… did the Berlin Wall stand?" as a "what was/did the Berlin Wall …?" query and falls back to extracting the first salient predicate from the context. Even if the question were well formed, the correct answer (28) is not present in the context as a span (it has to be derived as 1989) 1961, and a 77M-parameter encoder–decoder like flan-t5-small is poor at multi step arithmetic on dates. So this is a combined failure. (i) A defective question that misleads the parser into treating it as a definitional query. (ii) A small model that cannot perform the date subtraction it would need even if the parse were right.


In [14]:
context="Joseph Robinette Biden Jr.(born November 20, 1942) is an American politician who was the 46th president of the United States from 2021 to 2025. A member of the Democratic Party, he represented Delaware in the United States Senate from 1973 to 2009 and also served as the 47th vice president under President Barack Obama from 2009 to 2017."
question="Who was the president of United States during 2021-2025?"
vanilla_qa_pipe(f"CONTEXT: {context}\nQUESTION:{question}\nANSWER:")

'President Barack Obama'

A9.2 The correct answer (Joseph Robinette Biden Jr. or Biden) is present in the context, but it appears at the very start, detached from the phrase "from 2021 to 2025". The model has to bind the subject of the first sentence (Joseph Robinette Biden Jr.) to the role description in the relative clause (who was the 46th president of the United States from 2021 to 2025). The context then immediately introduces a second president by name (President Barack Obama) preceded by the title "President", which is a stronger surface cue for an extractive QA model than the more distant relative-clause construction that mentions Biden. flan-t5-small therefore latches onto the closest "President X" pattern and answers "Barack Obama". This is a classic distractor failure. Even with the correct context, a small model picks the answer with the strongest local lexical signal rather than the one supported by the semantics.


#END OF ASSIGNMENT
